In [ ]:
# kaggle para obter a base de dados
import kagglehub

# manipular dados
import numpy as np
import pandas as pd

# vizualizar dados
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# IA
import sklearn
from sklearn import preprocessing

import shutil
import os

In [ ]:
path = kagglehub.dataset_download("uciml/iris")

In [ ]:
print(f"O conjunto de dados foi baixado em: {path}")

In [ ]:
destino = "./data"
os.makedirs(destino, exist_ok=True)
shutil.copytree(path, destino, dirs_exist_ok=True)
print("Dataset copiado para:", os.path.abspath(destino))

In [ ]:
df = pd.read_csv("data/Iris.csv")
df.head(3)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.nunique()

In [ ]:
df.duplicated().sum()

Agora que vimos que os dados não precisam ser tratados primordialmente, podemos vizualizar esses dados em gráficos para ter uma maior compreensão.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

axes[0].hist(df['SepalWidthCm'], bins=20, color="skyblue", edgecolor="black")
axes[0].set_title("Histograma SepalWidthCm")
axes[0].set_xlabel("Valor")
axes[0].set_ylabel("Frequência")

axes[1].hist(df['SepalLengthCm'], bins=20, color="skyblue", edgecolor="black")
axes[1].set_title("Histograma SepalLengthCm")
axes[1].set_xlabel("Valor")
axes[1].set_ylabel("Frequência")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

axes[0].hist(df['PetalWidthCm'], bins=20, color="skyblue", edgecolor="black")
axes[0].set_title("Histograma PetalWidthCm")
axes[0].set_xlabel("Valor")
axes[0].set_ylabel("Frequência")

axes[1].hist(df['PetalLengthCm'], bins=20, color="skyblue", edgecolor="black")
axes[1].set_title("Histograma PetalLengthCm")
axes[1].set_xlabel("Valor")
axes[1].set_ylabel("Frequência")

plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(df[["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]])
plt.show()

In [ ]:
colunas_float = df.select_dtypes("float64").columns.to_list()
df_float = df[colunas_float]

# Estatísticas básicas das variáveis numéricas
df_float.describe().T
plt.figure(figsize=(8,6))
sns.heatmap(df_float.corr(), annot=True, cmap="coolwarm", center=0)
plt.title("Matriz de Correlação")
plt.show()

Com essa análise minuciosa das colunas disponíveis, percebe-se que se tem apenas 4 colunas e que todas elas parecem estar correlacionadas com nosso alvo "ESPÉCIE". Assim, decidimos seguir utilizando todas colunas disponíveis.

A coluna ID não agrega nada em relação quanto a espécie, porém, não é ideal eliminá-la, pois é necessario ter algum atributo que indentifique cada planta uníca.

-------------------------------------------------------------------

Agora que já vimos a relação entre cada variável, partimos para o mais importante: vizualizar se esses atributos tem impacto claro na espécie.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

sns.scatterplot(data=df, x="SepalLengthCm", y="SepalWidthCm", hue="Species", ax=axes[0])
axes[0].set_title("Relação entre SepalLength e SepalWidth por Espécie")

sns.scatterplot(data=df, x="PetalLengthCm", y="PetalWidthCm", hue="Species", ax=axes[1])
axes[1].set_title("Relação entre PetalLength e PetalWidth por Espécie")

plt.show()

É bem claro, principalmente no segundo gráfico, a **relação forte** que se tem entre a espécie da planta e os atributos, no caso **"PetalLength" e "PetalWidth"**.
Dessa forma, entendemos que eles serão os atributos principais de decisão em nosso projeto de Machine Learning

### Preparação e Divisão dos Dados

In [ ]:
# Aplicando Label-Encoding na coluna 'Species'
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Species_encoded'] = le.fit_transform(df['Species'])

display(df.head(5))

Optei pelo uso de Label Encoding ao invés de One-Hot Encoding por conta da coluna alvo (Species) possuir 3 valores

In [ ]:
df.info()

### Previsão com Random Forest

In [ ]:
# Imports necessários
from sklearn.tree import DecisionTreeClassifier
from sklearn import model_selection
from sklearn.metrics import confusion_matrix,classification_report,accuracy_score,roc_auc_score

In [ ]:
# Definindo a variável target
target = 'Species_encoded'

feature_col_arvore = df.columns.to_list()
feature_col_arvore.remove('Species')
feature_col_arvore.remove('Species_encoded')

x = df[feature_col_arvore]
y = df[target]

In [ ]:
# Array para armazenar a acurácia de cada fold
acc_Dtree=[]
kf=model_selection.StratifiedKFold(n_splits=5)
for fold , (trn_,val_) in enumerate(kf.split(X=x,y=y)):

    X_train=df.loc[trn_,feature_col_arvore]
    y_train=df.loc[trn_,target]

    X_valid=df.loc[val_,feature_col_arvore]
    y_valid=df.loc[val_,target]

    clf=DecisionTreeClassifier(criterion="entropy", max_depth=5, min_samples_leaf=5)
    clf.fit(X_train,y_train)
    y_pred_proba=clf.predict_proba(X_valid) # Usamos predict_proba para obter as probabilidades
    print(f"Fold : {fold} : ")
    # Note: classification_report expects predicted labels, not probabilities, so keep clf.predict() for it
    y_pred = clf.predict(X_valid)
    print(classification_report(y_valid,y_pred))
    acc=roc_auc_score(y_valid,y_pred_proba, multi_class='ovr') # Use the probabilities for roc_auc_score
    acc_Dtree.append(acc)
    print(f"Acurácia para {fold+1} : {acc}")

In [ ]:
media_acc = np.mean(acc_Dtree)

print(f'A acurácia média é : {media_acc}')

In [ ]:
import graphviz
from sklearn import tree

dot_data = tree.export_graphviz(clf, out_file=None,
                                feature_names=feature_col_arvore,
                                class_names=['0', '1', '2'],
                                filled=True, rounded=True)

# Mostra a árvore
graph = graphviz.Source(dot_data, format="png")
graph

### Previsão Com KNN

In [ ]:
#importa a função do knn
from sklearn.neighbors import KNeighborsClassifier
#separar features e label
features = df[['PetalLengthCm', 'PetalWidthCm']]
label = df['Species_encoded']

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

# Separar os dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(features, label, test_size=0.25, random_state=42, stratify=label)

# Definindo os hiperparâmetros
param_grid = {'n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15]}

# Criar o modelo KNN
knn = KNeighborsClassifier()

# Configurar o GridSearchCV
# cv=5 indica validação cruzada com 5 folds
grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='accuracy')

# Executar a busca na grade usando os dados de treino
grid_search.fit(X_train, y_train)

# Exibir os melhores hiperparâmetros encontrados
print("Melhores hiperparâmetros:", grid_search.best_params_)

# Exibir a melhor pontuação (acurácia média da validação cruzada)
print("Melhor acurácia na validação cruzada:", grid_search.best_score_)

# Avaliar o modelo com os melhores hiperparâmetros nos dados de teste
best_knn = grid_search.best_estimator_
accuracy_test = best_knn.score(X_test, y_test)
print("Acurácia nos dados de teste:", accuracy_test)

In [ ]:
#cria o modelo com k igual ao melhor n_neighbors encontrado
knn = KNeighborsClassifier(n_neighbors=best_knn.n_neighbors)

#treina o modelo com os dados acima
knn.fit(features, label)

In [ ]:
#input dos x e y digitados
new_x = float(input("Petal Length: "))
new_y = float(input("Petal Width : "))

#transforma o x e y numa tupla
new_point = [[new_x, new_y]] # KNN expects a 2D array for prediction

#o modelo prediz a classe do novo ponto
prediction_encoded = knn.predict(new_point)

# Mapear a previsão numérica de volta para o nome da espécie
# Certifique-se que 'le' (LabelEncoder) está acessível neste escopo
predicted_species = le.inverse_transform(prediction_encoded)

# Extrair os valores das colunas para plotagem
x_values = df['PetalLengthCm'].values
y_values = df['PetalWidthCm'].values
species_names = df['Species'].values # Usar os nomes das espécies originais para plotagem

# Plota os pontos existentes, colorindo por espécie e adicionando labels para a legenda
unique_species = np.unique(species_names)
colors = plt.cm.viridis(np.linspace(0, 1, len(unique_species))) # Gerar cores distintas

for i, species in enumerate(unique_species):
    mask = species_names == species
    plt.scatter(x_values[mask], y_values[mask], c=[colors[i]], label=species, edgecolors='black', alpha=0.6)

# Plota o novo ponto em vermelho
plt.scatter(new_x, new_y, c='red', s=200, label=f'New Point ({predicted_species[0]})', edgecolors='black', marker='*') # Use um marcador diferente para o novo ponto

# Adiciona texto para o novo ponto
# plt.text(x=new_x + 0.1, y=new_y + 0.1, s=f"New Point") # Pode remover se a legenda for clara

plt.xlabel("Petal Length (Cm)")
plt.ylabel("Petal Width (Cm)")
plt.title("KNN Classification with New Point")
plt.legend() # Mostra a legenda
plt.grid(True) # Adiciona um grid
plt.show()

print(f'\nCom base na classificação do KNN com K={best_knn.n_neighbors}, uma planta com PetalLength={new_x} e PetalWidth={new_y}, a planta nova seria uma {predicted_species[0]}')